# === Cell 1: Title (Markdown) ===
# ## 00_setup_and_dqn_clean
# Clean notebook: installs, mounts Drive, creates env, DQN training (MLP), saves expert dataset and GIFs.

In [1]:
# === Cell 2: Install & Mount Drive ===
!pip install --quiet minatar imageio-ffmpeg
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/rl-final-project"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "dqn"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "random_breakout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "expert_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "visuals", "bc_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_DIR, "imitation"), exist_ok=True)
print('Drive mounted and project directories ready:', DRIVE_PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and project directories ready: /content/drive/MyDrive/rl-final-project


In [2]:
# === Cell 3: Imports & device ===
import random
import time
import pickle
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [3]:

# === Cell 4: Create MinAtar env and quick check ===
import minatar
env = minatar.Environment('breakout')
print('env.state_shape() =', env.state_shape())
print('env.num_actions() =', env.num_actions())


env.state_shape() = [10, 10, 4]
env.num_actions() = 6


In [4]:
# === Cell 5: Helper functions: preprocess_state, state_to_numpy ===
import numpy as _np

def preprocess_state(st):
    arr = _np.array(st, dtype=_np.float32)
    # handle unexpected shapes defensively
    if arr.ndim == 1:
        # try to reshape if possible
        if arr.size == 1:
            arr = _np.full((10,10), arr[0], dtype=_np.float32)
        else:
            try:
                arr = arr.reshape(10,10)
            except Exception:
                arr = _np.zeros((10,10), dtype=_np.float32)
    if arr.ndim == 3:
        arr = _np.sum(arr, axis=2)
    mn, mx = arr.min(), arr.max()
    rng = mx - mn if mx > mn else 1.0
    arr = (arr - mn) / rng
    flat = arr.flatten()
    if flat.shape[0] != 100:
        flat = _np.resize(flat, 100)
    return flat

def state_to_numpy(st):
    return _np.array(st, dtype=_np.float32)

print('preprocess_state ok ->', preprocess_state(env.reset()).shape)

preprocess_state ok -> (100,)


In [5]:


# === Cell 6: ReplayBuffer & QNetworkMLP ===
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    def push(self, *args):
        self.buffer.append(Transition(*args))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))
    def __len__(self):
        return len(self.buffer)

class QNetworkMLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, n_actions):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, n_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print('Network defined.')

Network defined.


In [6]:

# === Cell 7: Utilities ===
def linear_eps(step, eps_start, eps_final, eps_decay):
    return eps_final + (eps_start - eps_final) * max(0, (1 - step / eps_decay))

def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)

In [7]:



# === Cell 8: Training function (clean) ===
import time

def train_dqn_to_drive(env,
                       drive_dir,
                       num_steps=100000,
                       buffer_capacity=50000,
                       batch_size=64,
                       gamma=0.99,
                       lr=1e-3,
                       target_update_freq=1000,
                       start_learning=1000,
                       eps_start=1.0,
                       eps_final=0.05,
                       eps_decay=40000,
                       hidden_sizes=[256,128],
                       save_every=5000,
                       reset_every=500):
    dqn_dir = os.path.join(drive_dir, 'dqn')
    plots_dir = os.path.join(drive_dir, 'plots')
    os.makedirs(dqn_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)
    model_path = os.path.join(dqn_dir, 'best_model.pth')
    expert_dataset_path = os.path.join(dqn_dir, 'expert_dataset.pkl')
    rewards_plot_path = os.path.join(plots_dir, 'dqn_rewards.png')

    env.reset()
    input_dim = preprocess_state(env.state()).shape[0]
    n_actions = env.num_actions()

    q_net = QNetworkMLP(input_dim, hidden_sizes, n_actions).to(device)
    q_target = QNetworkMLP(input_dim, hidden_sizes, n_actions).to(device)
    q_target.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    replay = ReplayBuffer(buffer_capacity)

    all_steps = 0
    episode_reward = 0.0
    episode_rewards = []
    rewards_log = []
    expert_transitions = []
    losses = []
    best_avg = -float('inf')

    state = preprocess_state(env.reset())

    while all_steps < num_steps:
        eps = linear_eps(all_steps, eps_start, eps_final, eps_decay)
        if random.random() < eps:
            action = random.randrange(n_actions)
        else:
            with torch.no_grad():
                s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                qvals = q_net(s_t)
                action = int(torch.argmax(qvals, dim=1).item())

        reward, done = env.act(action)
        next_state = preprocess_state(env.state())

        # debug shapes
        if len(state) != input_dim:
            print('BAD STATE SHAPE:', len(state))
        if len(next_state) != input_dim:
            print('BAD NEXT STATE SHAPE:', len(next_state))

        replay.push(state, action, reward, next_state, float(done))
        expert_transitions.append((state, action, reward, next_state, float(done)))

        episode_reward += reward
        all_steps += 1
        state = next_state

        if all_steps % reset_every == 0:
            episode_rewards.append(episode_reward)
            rewards_log.append(episode_reward)
            episode_reward = 0.0
            env.reset()
            state = preprocess_state(env.state())

        if len(replay) > start_learning:
            batch = replay.sample(batch_size)
            # ensure shapes
            try:
                states = torch.tensor(np.stack(batch.state), dtype=torch.float32).to(device)
            except Exception as e:
                print('STACK ERROR:', e)
                print('batch.state shapes:', [s.shape for s in batch.state])
                raise

            actions = torch.tensor(batch.action, dtype=torch.long).unsqueeze(1).to(device)
            rewards_t = torch.tensor(batch.reward, dtype=torch.float32).unsqueeze(1).to(device)
            next_states = torch.tensor(np.stack(batch.next_state), dtype=torch.float32).to(device)
            dones = torch.tensor(batch.done, dtype=torch.float32).unsqueeze(1).to(device)

            q_values = q_net(states).gather(1, actions)
            with torch.no_grad():
                q_next = q_target(next_states).max(1)[0].unsqueeze(1)
                q_target_val = rewards_t + gamma * (1 - dones) * q_next

            loss = nn.functional.mse_loss(q_values, q_target_val)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        if all_steps % target_update_freq == 0:
            q_target.load_state_dict(q_net.state_dict())

        if all_steps % save_every == 0:
            avg_recent = float(np.mean(rewards_log[-10:])) if len(rewards_log) >= 1 else 0.0
            print(f"[{time.strftime('%H:%M:%S')}] Step {all_steps}/{num_steps} eps={eps:.3f} avg_recent={avg_recent:.3f}")
            if avg_recent > best_avg:
                best_avg = avg_recent
                torch.save(q_net.state_dict(), model_path)
                print('Saved improved model to', model_path)

    torch.save(q_net.state_dict(), model_path)
    print('Saved final model:', model_path)
    save_pickle(expert_transitions, expert_dataset_path)
    print('Saved expert dataset:', expert_dataset_path)

    if len(episode_rewards) > 0:
        plt.figure(figsize=(8,4))
        plt.plot(episode_rewards)
        plt.title('DQN episode rewards (manual reset chunks)')
        plt.grid(True)
        plt.savefig(rewards_plot_path, dpi=150)
        plt.close()

    return {'model_path': model_path, 'expert_dataset_path': expert_dataset_path, 'rewards_plot_path': rewards_plot_path}



In [8]:
res = train_dqn_to_drive(
    env,
    DRIVE_PROJECT_DIR,
    num_steps=50000,   # или сначала тест: 5000
    buffer_capacity=50000,
    batch_size=64,
    save_every=10000
)
print(res)


[07:31:57] Step 10000/50000 eps=0.763 avg_recent=0.800
Saved improved model to /content/drive/MyDrive/rl-final-project/dqn/best_model.pth
[07:32:22] Step 20000/50000 eps=0.525 avg_recent=0.400
[07:32:51] Step 30000/50000 eps=0.288 avg_recent=0.700
[07:33:19] Step 40000/50000 eps=0.050 avg_recent=0.400
[07:33:47] Step 50000/50000 eps=0.050 avg_recent=0.300
Saved final model: /content/drive/MyDrive/rl-final-project/dqn/best_model.pth
Saved expert dataset: /content/drive/MyDrive/rl-final-project/dqn/expert_dataset.pkl
{'model_path': '/content/drive/MyDrive/rl-final-project/dqn/best_model.pth', 'expert_dataset_path': '/content/drive/MyDrive/rl-final-project/dqn/expert_dataset.pkl', 'rewards_plot_path': '/content/drive/MyDrive/rl-final-project/plots/dqn_rewards.png'}


In [11]:
def create_expert_rollout_gif(env, model_path, save_subdir="visuals/expert_rollout", n_steps=400, scale=20):
    save_dir = os.path.join(DRIVE_PROJECT_DIR, save_subdir)
    os.makedirs(save_dir, exist_ok=True)

    input_dim = preprocess_state(env.reset()).shape[0]
    n_actions = env.num_actions()
    expert_net = QNetworkMLP(input_dim, [256,128], n_actions).to(device)
    expert_net.load_state_dict(torch.load(model_path, map_location=device))
    expert_net.eval()

    def expert_policy(state):
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            q = expert_net(s)
            return int(torch.argmax(q, dim=1).item())

    frames = []
    env.reset()
    state = preprocess_state(env.state())

    for _ in range(n_steps):
        action = expert_policy(state)
        reward, done = env.act(action)

        st = state_to_numpy(env.state())
        img = st.sum(axis=2) if st.ndim == 3 else st.squeeze()

        mn, mx = img.min(), img.max()
        rng = mx - mn if mx > mn else 1.0
        img_norm = ((img - mn) / rng * 255).astype(np.uint8)

        up = Image.fromarray(img_norm).resize(
            (img_norm.shape[0]*scale, img_norm.shape[1]*scale),
            Image.NEAREST
        )
        frames.append(np.array(up))

        state = preprocess_state(env.state())

    gif_path = os.path.join(save_dir, "expert_rollout.gif")
    imageio.mimsave(gif_path, frames, fps=12)

    print("Saved expert rollout gif:", gif_path)
    return gif_path


In [12]:
model_path = os.path.join(DRIVE_PROJECT_DIR, "dqn", "best_model.pth")
gif_path = create_expert_rollout_gif(env, model_path)
gif_path


Saved expert rollout gif: /content/drive/MyDrive/rl-final-project/visuals/expert_rollout/expert_rollout.gif


'/content/drive/MyDrive/rl-final-project/visuals/expert_rollout/expert_rollout.gif'